### **Configuração Pasta Raiz**

In [1]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))

# Adiciona a raiz do projeto ao path do Python para que as importações funcionem
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

### **Bibliotecas**

In [2]:
import pandas as pd
import src.utils.features as features

### **Leitura da base de dados**

In [3]:
filename = "01_final_dataset_merged.csv"

df_merged = pd.read_csv(f'../data/{filename}')

### **Criando variável resposta**

In [10]:
DRAFT_LIMIT_YEAR = 2021

In [4]:
df_merged.sort_values(by=['vorp_5y_sum'], ascending=[False])
vorp_5y_sum = df_merged['vorp_5y_sum']
vorp_5y_sum.describe()

count    1542.000000
mean        1.102918
std         3.263285
min        -4.100000
25%        -0.200000
50%         0.000000
75%         0.800000
max        32.000000
Name: vorp_5y_sum, dtype: float64

In [5]:
df_merged.sort_values(by=['vorp_5y_sum'], ascending=[False])

,player_id,player_name,draft_year,college_name,draft_pick,vorp_5y_sum,seasons_played,draft_age,true_shooting_pct,three_point_attempt_rate,...,three_point_pct,career_games,career_starts,career_minutes,career_awards_count,years_college,position,height_cm,weight_kg,mock_pick
292,paulch01,Chris Paul,2005,Wake Forest,4,32.0,5.0,19.0,0.601,0.306,...,0.474,63.0,62.0,2110.0,1.0,2.0,Guard,182.2,80.7,4.0
410,duranke01,Kevin Durant,2007,Texas,2,23.7,5.0,18.0,0.587,0.314,...,0.404,35.0,35.0,1255.0,1.0,1.0,Forward,205.7,97.5,2.0
709,davisan02,Anthony Davis,2012,Kentucky,1,21.4,5.0,18.0,0.654,0.059,...,0.150,40.0,40.0,1281.0,1.0,1.0,Forward,206.4,100.6,1.0
176,wadedw01,Dwyane Wade,2003,Marquette,5,21.1,5.0,21.0,0.573,0.088,...,0.318,65.0,65.0,1993.0,1.0,3.0,Guard,192.4,96.2,7.0
889,townska01,Karl-Anthony Towns,2015,Kentucky,1,21.1,5.0,19.0,0.627,0.031,...,0.250,39.0,39.0,822.0,1.0,1.0,Forward,210.8,113.4,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1048,bacondw01,Dwayne Bacon,2017,Florida State,40,-3.1,4.0,21.0,0.544,0.350,...,0.333,69.0,67.0,1987.0,0.0,2.0,Guard,194.9,100.5,59.0
422,thornal01,Al Thornton,2007,Florida State,14,-3.2,5.0,23.0,0.616,0.176,...,0.444,124.0,69.0,2723.0,0.0,4.0,Guard,197.5,100.2,14.0
1053,brookdi01,Dillon Brooks,2017,Oregon,45,-3.6,5.0,21.0,0.585,0.336,...,0.401,109.0,98.0,3153.0,2.0,3.0,Forward,195.6,99.9,47.0
6,mihmch01,Chris Mihm,2000,Texas,7,-3.8,5.0,20.0,0.578,0.038,...,0.467,96.0,93.0,2810.0,1.0,3.0,Center,213.4,120.2,5.0


In [11]:
primeira_exclusao = df_merged[(df_merged['college_name'].notna())].copy()
segunda_exclusao = primeira_exclusao[(primeira_exclusao['career_games'].notna())].copy()
terceira_exclusao = segunda_exclusao[(segunda_exclusao['draft_year'] < DRAFT_LIMIT_YEAR)].copy()

print(f"Total de jogadores: {len(df_merged)}")
print(f"Total de jogadores após a primeira exclusão: {len(primeira_exclusao)}")
print(f"Total de jogadores após a segunda exclusão: {len(segunda_exclusao)}")
print(f"Total de jogadores após a terceira exclusão: {len(terceira_exclusao)}")


Total de jogadores: 1542
Total de jogadores após a primeira exclusão: 1194
Total de jogadores após a segunda exclusão: 1183
Total de jogadores após a terceira exclusão: 954


In [13]:

target_map = {
    3: 'Star',
    2: 'Starter',
    1: 'Rotation',
    0: 'Bust'
}

df_vorp_target = df_merged[(df_merged['college_name'].notna()) & (df_merged['career_games'].notna())].copy()
# df_vorp_target = df_merged[(df_merged['draft_year'] > DRAFT_LIMIT_YEAR) & (df_merged['college_name'].notna()) & (df_merged['career_games'].notna())].copy()


# Aplicando ao seu dataframe alvo (supondo que já tenha criado as colunas que discutimos antes)
df_vorp_target['target_class'] = df_vorp_target.apply(features.categorize_player, axis=1)
df_vorp_target['target_label'] = df_vorp_target['target_class'].map(target_map)

# Verificação da distribuição (MUITO IMPORTANTE)
counts = df_vorp_target['target_label'].value_counts()
percents = df_vorp_target['target_label'].value_counts(normalize=True) * 100

# 2. Criando o DataFrame da tabela
tabela_resumo = pd.DataFrame({
    'Frequência Absoluta': counts,
    'Frequência Relativa (%)': percents.map('{:.2f}%'.format)
})

print(tabela_resumo)

              Frequência Absoluta Frequência Relativa (%)
target_label                                             
Bust                          665                  56.21%
Rotation                      325                  27.47%
Starter                       140                  11.83%
Star                           53                   4.48%


In [8]:
df_vorp_target.to_csv(PROJECT_ROOT + "/data/02_college_players_dataset.csv", index=False, encoding="utf-8-sig")

### **Reduzir o número de nomes de Colleges e Conferences**

In [9]:
top_items  = [
    {'col': 'college_name', 'small_label': 'Small_College', 'size_threshold': 20},
    {'col': 'conference', 'small_label': 'Small_Conference', 'size_threshold': 10},
]

for item in top_items:
    col = item['col']
    small_label = item['small_label']
    size_threshold = item.get('size_threshold', 20) 
    top = df_vorp_target[col].value_counts().head(size_threshold).index

    df_vorp_target[f'{col}_new'] = df_vorp_target[col].apply(lambda x: x if x in top else f'{small_label}')

    df_vorp_target.drop(col, axis=1, inplace=True)
    df_vorp_target.rename(columns={f'{col}_new': col}, inplace=True)



### **Dividir a base**

In [10]:
df_vorp_target_before_2021 = df_vorp_target[(df_vorp_target['draft_year'] < DRAFT_LIMIT_YEAR)].copy()
df_vorp_target_before_2021.to_csv(PROJECT_ROOT + "/data/03_college_players_before_2021_dataset.csv", index=False, encoding="utf-8-sig")

In [11]:
df_vorp_target_after_2021 = df_vorp_target[(df_vorp_target['draft_year'] >= DRAFT_LIMIT_YEAR)].copy()
df_vorp_target_after_2021.to_csv(PROJECT_ROOT + "/data/04_college_players_after_2021_dataset.csv", index=False, encoding="utf-8-sig")

### **Drop colunas**

In [12]:
drop_columns = [
    'draft_pick', 'vorp_5y_sum','seasons_played','target_class'
]

for col in drop_columns:
    df_vorp_target_before_2021.drop(col, axis=1, inplace=True)
    df_vorp_target_after_2021.drop(col, axis=1, inplace=True)

In [13]:
df_vorp_target_before_2021.to_csv(PROJECT_ROOT + "/data/00_final_dataset.csv", index=False, encoding="utf-8-sig")
df_vorp_target_after_2021.to_csv(PROJECT_ROOT + "/data/05_college_players_after_2021_dropped_dataset.csv", index=False, encoding="utf-8-sig")